# 08 — Train QLSTM

**PLTMH–ELC–QLSTM**

Notebook ini mendokumentasikan dan memverifikasi tahap QLSTM
*hybrid quantum–classical* setelah baseline LSTM dibekukan.

Tahap yang dicakup:

- feasibility benchmark;
- architecture freeze;
- parameter audit;
- training/validation freeze;
- single frozen test evaluation;
- batas klaim ilmiah sebelum closed-loop.

Notebook ini **tidak melakukan post-test tuning**.


## 1. Catatan Rekonstruksi Implementasi

File repository:

`src/models/qlstm.py`

dan:

`src/models/vqc.py`

masih berupa placeholder dari struktur proyek awal.

Karena itu notebook migrasi **tidak mengarang ulang** implementasi
quantum circuit hanya dari metadata arsitektur.

Source asli Cell 146, 147, 148, 149, 150, dan 150-FIX telah
disimpan sebagai bukti provenance pada:

`data/recovery/cell165_qlstm_runtime_source_snapshot.json`

Snapshot tersebut merupakan source input asli dari runtime Colab,
bukan hasil rekonstruksi semantik.

Mode default Notebook 08 adalah **verifikasi frozen evidence**.


## 2. Kebijakan Ilmiah QLSTM

Arsitektur QLSTM dibekukan **sebelum** validation/test digunakan
untuk memilih arsitektur.

Validation hanya digunakan untuk early stopping setelah
arsitektur dibekukan. Test set dievaluasi satu kali setelah training
selesai.

Hasil test tersebut tidak digunakan untuk:

- mengubah jumlah qubit;
- mengubah depth VQC;
- mengubah regression head;
- mengubah learning rate;
- melatih ulang QLSTM;
- memilih ulang dataset atau split.

Hasil negatif dipertahankan.

**Tidak ada klaim quantum advantage.**


In [ ]:
# ============================================================
# 08.1 — PROJECT + FROZEN QLSTM ARTIFACT INTEGRITY
# ============================================================

from pathlib import Path

import hashlib
import json
import sys

import numpy as np
import pandas as pd


EXPECTED_UPSTREAM_CHECKPOINT = "8c002708ad90b5536aab6cbc5a3b3cd9cff89f50"

EXPECTED_NOTEBOOK_07_SHA256 = (
    "2b7efaebbfed7eb00062eabf0f1cba4a75b87134e9d5662e9778ca09ecda684b"
)

EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256 = (
    "2957710dc17b08e96b519f8fa9fadb7f798e012648ad1b9e310ad6d0a0f35742"
)

EXPECTED_QLSTM_ARTIFACT_SHA256 = (
    {
    "data/audits/cell144_qlstm_environment_readiness.json": "73a19c317d038db545ca98c38ac67511af14c649555e32cbb97221513f8a7384",
    "data/audits/cell146_qlstm_runtime_benchmark.csv": "97c70e44d464e3cfc871f247e80f1dcae8c521f9fe97b338d79326e280df26af",
    "data/audits/cell146_qlstm_runtime_benchmark.json": "363e01bfb303c28a5db005d0d70018178e59e9881a12342b162cc882c0d7e5f3",
    "data/audits/cell149_test_family_audit.csv": "ef58c956070a2387b4ed654ced13384534c8cdbe5b77729b4ac4e14f96bd177e",
    "data/audits/cell150_final_model_stage_audit.csv": "fe3849c7b44083eff145ea094312c59a56d51f52e87b3c77efaae011cf630daf",
    "data/audits/cell150_qlstm_test_regime_confusion.csv": "e8256f3c207f8a50c774046fe95cd88375ba1ed81892341635afe4be1b8aafb0",
    "data/audits/cell150_scientific_claim_freeze.csv": "67d7a59e4e63e014307b6f374fb73371283ac982154485783af2544171d43601",
    "data/recovery/cell150_pre_closed_loop_resume_notes.txt": "566a03546439c5cc7f0712df3ad4f7d58af7987279f5232ab07245ab3be4dcf8",
    "data/recovery/cell150_scientific_interpretation_freeze.json": "ec1db4cf08a247451fad8199078a74386865c9f1a7b3883b97575a092c743a9e",
    "models/qlstm/cell147_parameter_audit.csv": "29ce108678cb69cb2f8ff9e67327324a87ca375813573e6a4bc8f13a728d6f4d",
    "models/qlstm/cell147_qlstm_architecture.json": "a72ced2b5f0490fa2d392900bf31766ed6456f25a615b7499add65ee41c0b8cd",
    "models/qlstm/cell147_qlstm_initial_state.pt": "1935f7048222a52ff2a6a9cb36ea41dfabfae3e7f8c8d5e31fec7dabe0dcf1b2",
    "models/qlstm/cell147_smoke_test.json": "7b45a6d13dbbe957332df008adee1211643f6c5e3b0b7d29e6f9142f0ce9df8f",
    "models/qlstm/cell148_qlstm_best.pt": "bf1a049a8bae552f5be7a3460507fe9d3912af7540ba617ea79623676862cfd7",
    "models/qlstm/cell148_training_history.csv": "b57b56d98b139abcac3e54f7f8b3b05601c3d30866400faa68d2916eaec344d0",
    "models/qlstm/cell148_training_metadata.json": "9cc2f8bee31cf274ec00501c0f9eb2f3471704273e8fca30cff7adc1c2ec359d",
    "models/qlstm/cell148_validation_metrics.csv": "aebd20a3d1e27f5ab08059201ccc79dec3c0bbab432c6610b20e9f8fc3f56e59",
    "models/qlstm/cell148_validation_predictions.csv.gz": "8a269b8ed284b97c90bc24569b4871ee00865ceca817a3b9bf9ba5adbf71b63b",
    "models/qlstm/cell149_frozen_model_test_comparison.csv": "a0e2ae1f662e00821d795d4180bcd8312d8e8f568f1855a550fbae4edb16a45f",
    "models/qlstm/cell149_qlstm_test_metrics.csv": "fd7d2a189609fb4d2c801c924884683f40053de135f9ecf5e660f758a9a3fe16",
    "models/qlstm/cell149_qlstm_test_predictions.csv.gz": "949c93b3cdc310950e2e0672164c99c77df0397a51ca6651495d7a707419f826",
    "models/qlstm/cell149_test_evaluation_metadata.json": "723d24bbaac00473571201b71087c580bd0ead257c11d5ba4c17a85f880bf342"
}
)

CRITICAL_QLSTM_RELPATHS = (
    {
    "architecture": "models/qlstm/cell147_qlstm_architecture.json",
    "best_checkpoint": "models/qlstm/cell148_qlstm_best.pt",
    "initial_state": "models/qlstm/cell147_qlstm_initial_state.pt",
    "parameter_audit": "models/qlstm/cell147_parameter_audit.csv",
    "smoke_test": "models/qlstm/cell147_smoke_test.json",
    "test_comparison": "models/qlstm/cell149_frozen_model_test_comparison.csv",
    "test_metrics": "models/qlstm/cell149_qlstm_test_metrics.csv",
    "training_history": "models/qlstm/cell148_training_history.csv",
    "training_metadata": "models/qlstm/cell148_training_metadata.json",
    "validation_metrics": "models/qlstm/cell148_validation_metrics.csv",
    "validation_predictions": "models/qlstm/cell148_validation_predictions.csv.gz"
}
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "models").exists()
        and
        (candidate / "notebooks").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


NOTEBOOK_07_PATH = (
    PROJECT_ROOT
    /
    "notebooks"
    /
    "07_train_lstm.ipynb"
)


if (
    sha256_file(
        NOTEBOOK_07_PATH
    )
    !=
    EXPECTED_NOTEBOOK_07_SHA256
):

    raise RuntimeError(
        "Upstream Notebook 07 changed."
    )


RUNTIME_SOURCE_SNAPSHOT_PATH = (
    PROJECT_ROOT
    /
    "data"
    /
    "recovery"
    /
    "cell165_qlstm_runtime_source_snapshot.json"
)


if (
    sha256_file(
        RUNTIME_SOURCE_SNAPSHOT_PATH
    )
    !=
    EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256
):

    raise RuntimeError(
        "Archived QLSTM runtime source snapshot changed."
    )


artifact_integrity = {}


for relative_path, expected_sha in (
    EXPECTED_QLSTM_ARTIFACT_SHA256.items()
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    artifact_integrity[
        relative_path
    ] = bool(
        path.exists()
        and
        sha256_file(
            path
        )
        ==
        expected_sha
    )


for relative_path, status in (
    artifact_integrity.items()
):

    print(
        f"{relative_path:74s}: {status}"
    )


FROZEN_QLSTM_ARTIFACTS_VALID = all(
    artifact_integrity.values()
)


if not FROZEN_QLSTM_ARTIFACTS_VALID:

    raise RuntimeError(
        "One or more frozen QLSTM artifacts changed."
    )


print(
    "\nFROZEN_QLSTM_ARTIFACTS_VALID:",
    FROZEN_QLSTM_ARTIFACTS_VALID
)


In [ ]:
# ============================================================
# 08.2 — FROZEN QLSTM ARCHITECTURE AUDIT
# ============================================================

EXPECTED_ARCHITECTURE = {
    "architecture": {
        "entanglement": "ring_CNOT",
        "head": [
            4,
            64,
            60,
            16,
            2
        ],
        "hidden_dim": 4,
        "input_dim": 4,
        "n_qubits": 4,
        "quantum_encoding": "RY",
        "quantum_recurrent_gates": 4,
        "sequence_length": 11,
        "variational_gate": "Rot",
        "vqc_depth": 1
    },
    "architecture_selection_basis": "parameter_matching_and_cell146_compute_feasibility",
    "cell146_context": {
        "batch_256_feasible": true,
        "estimated_epoch_min": 0.5905182440666598,
        "qnode_invocations_epoch_forward": 1936
    },
    "full_training_started": false,
    "hyperparameter_search": false,
    "initial_state_path": "models/qlstm/cell147_qlstm_initial_state.pt",
    "initial_state_sha256": "1935f7048222a52ff2a6a9cb36ea41dfabfae3e7f8c8d5e31fec7dabe0dcf1b2",
    "parameter_audit": {
        "absolute_difference_pct": 0.07371913011426465,
        "classical_gate_projection_parameters": 144,
        "classical_lstm_parameters": 5426,
        "difference": -4,
        "qlstm_parameters": 5422,
        "quantum_parameters": 48,
        "regression_head_parameters": 5230
    },
    "stage": "FINAL_QLSTM_ARCHITECTURE_FREEZE",
    "test_used_for_architecture_selection": false,
    "training_protocol": {
        "batch_size": 256,
        "dtype": "torch.float64",
        "learning_rate": 0.001,
        "loss": "MSE",
        "max_epochs": 100,
        "min_delta": 1e-06,
        "optimizer": "Adam",
        "patience": 12,
        "seed": 20260909
    },
    "validation_used_for_architecture_selection": false
}


ARCHITECTURE_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_QLSTM_RELPATHS[
        "architecture"
    ]
)


architecture = json.loads(
    ARCHITECTURE_PATH.read_text(
        encoding="utf-8"
    )
)


ARCHITECTURE_EXACT = bool(
    architecture
    ==
    EXPECTED_ARCHITECTURE
)


if not ARCHITECTURE_EXACT:

    raise RuntimeError(
        "Frozen QLSTM architecture metadata changed."
    )


arch = architecture[
    "architecture"
]


parameter_audit = architecture[
    "parameter_audit"
]


print(
    "Input dimension                :",
    arch[
        "input_dim"
    ]
)

print(
    "Sequence length                :",
    arch[
        "sequence_length"
    ]
)

print(
    "Hidden dimension               :",
    arch[
        "hidden_dim"
    ]
)

print(
    "Qubits                         :",
    arch[
        "n_qubits"
    ]
)

print(
    "VQC depth                      :",
    arch[
        "vqc_depth"
    ]
)

print(
    "Quantum recurrent gates        :",
    arch[
        "quantum_recurrent_gates"
    ]
)

print(
    "Quantum encoding               :",
    arch[
        "quantum_encoding"
    ]
)

print(
    "Variational gate               :",
    arch[
        "variational_gate"
    ]
)

print(
    "Entanglement                   :",
    arch[
        "entanglement"
    ]
)

print(
    "Regression head                :",
    arch[
        "head"
    ]
)

print(
    "QLSTM parameters               :",
    parameter_audit[
        "qlstm_parameters"
    ]
)

print(
    "Classical LSTM parameters      :",
    parameter_audit[
        "classical_lstm_parameters"
    ]
)

print(
    "Quantum parameters             :",
    parameter_audit[
        "quantum_parameters"
    ]
)

print(
    "Classical gate projection      :",
    parameter_audit[
        "classical_gate_projection_parameters"
    ]
)

print(
    "Regression-head parameters     :",
    parameter_audit[
        "regression_head_parameters"
    ]
)

print(
    "Validation used for arch select:",
    architecture[
        "validation_used_for_architecture_selection"
    ]
)

print(
    "Test used for arch selection   :",
    architecture[
        "test_used_for_architecture_selection"
    ]
)

print(
    "\nARCHITECTURE_EXACT:",
    ARCHITECTURE_EXACT
)


In [ ]:
# ============================================================
# 08.3 — FROZEN QLSTM TRAINING / VALIDATION AUDIT
# ============================================================

EXPECTED_TRAINING_METADATA = {
    "architecture_changed_during_training": false,
    "architecture_frozen_by": "CELL_147",
    "classical_validation_context": {
        "cell139_lstm_validation_mse": 0.517013073,
        "cell143_mean_endpoint_mlp_validation_mse": 0.5258254733085632,
        "cell143_mean_lstm_validation_mse": 0.4991619627475738,
        "qlstm_improvement_vs_cell139_lstm_pct": -5.3444850190804285,
        "qlstm_improvement_vs_cell143_mean_lstm_pct": -9.111831405430868
    },
    "dataset_access": {
        "test": false,
        "train": true,
        "validation": true
    },
    "device": "cpu",
    "dtype": "torch.float64",
    "hyperparameter_search_performed": false,
    "model": {
        "best_model_sha256": "bf1a049a8bae552f5be7a3460507fe9d3912af7540ba617ea79623676862cfd7",
        "initial_state_sha256": "1935f7048222a52ff2a6a9cb36ea41dfabfae3e7f8c8d5e31fec7dabe0dcf1b2",
        "quantum_parameter_fraction_pct": 0.885282183696053,
        "quantum_parameters": 48,
        "total_parameters": 5422
    },
    "scientific_policy": {
        "next_cell": "CELL_149_FROZEN_MODEL_TEST_EVALUATION",
        "test_evaluation_performed": false,
        "test_used_for_model_selection": false,
        "validation_used_for_early_stopping": true
    },
    "seed": 20260909,
    "stage": "FULL_QLSTM_TRAINING_COMPLETE",
    "training": {
        "batch_size": 256,
        "best_epoch": 6,
        "best_train_mse": 0.3934475580172407,
        "best_validation_mse": 0.5446447592331723,
        "early_stopped": true,
        "epochs_run": 18,
        "learning_rate": 0.001,
        "max_epochs": 100,
        "min_delta": 1e-06,
        "optimizer": "Adam",
        "patience": 12,
        "runtime_s": 762.4303830900001
    },
    "validation_metrics": {
        "ki_mae": 1.467056118155746,
        "ki_rmse": 1.53948706958036,
        "kp_mae": 0.36821312355030744,
        "kp_rmse": 0.3866790529323389,
        "nearest_regime_accuracy": 0.5,
        "standardized_mse": 0.5446447592331723
    }
}


TRAINING_METADATA_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_QLSTM_RELPATHS[
        "training_metadata"
    ]
)


training_metadata = json.loads(
    TRAINING_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


TRAINING_METADATA_EXACT = bool(
    training_metadata
    ==
    EXPECTED_TRAINING_METADATA
)


if not TRAINING_METADATA_EXACT:

    raise RuntimeError(
        "Frozen QLSTM training metadata changed."
    )


INITIAL_STATE_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_QLSTM_RELPATHS[
        "initial_state"
    ]
)


BEST_MODEL_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_QLSTM_RELPATHS[
        "best_checkpoint"
    ]
)


initial_hash_ok = bool(
    sha256_file(
        INITIAL_STATE_PATH
    )
    ==
    training_metadata[
        "model"
    ][
        "initial_state_sha256"
    ]
)


best_hash_ok = bool(
    sha256_file(
        BEST_MODEL_PATH
    )
    ==
    training_metadata[
        "model"
    ][
        "best_model_sha256"
    ]
)


if not (
    initial_hash_ok
    and
    best_hash_ok
):

    raise RuntimeError(
        "QLSTM checkpoint hash contract failed."
    )


print(
    "Device                        :",
    training_metadata[
        "device"
    ]
)

print(
    "dtype                         :",
    training_metadata[
        "dtype"
    ]
)

print(
    "Epochs run                    :",
    training_metadata[
        "training"
    ][
        "epochs_run"
    ]
)

print(
    "Best epoch                    :",
    training_metadata[
        "training"
    ][
        "best_epoch"
    ]
)

print(
    "Best train MSE                :",
    training_metadata[
        "training"
    ][
        "best_train_mse"
    ]
)

print(
    "Best validation MSE           :",
    training_metadata[
        "training"
    ][
        "best_validation_mse"
    ]
)

print(
    "Training runtime [s]          :",
    training_metadata[
        "training"
    ][
        "runtime_s"
    ]
)

print(
    "Total parameters              :",
    training_metadata[
        "model"
    ][
        "total_parameters"
    ]
)

print(
    "Quantum parameters            :",
    training_metadata[
        "model"
    ][
        "quantum_parameters"
    ]
)

print(
    "Quantum parameter fraction [%]:",
    training_metadata[
        "model"
    ][
        "quantum_parameter_fraction_pct"
    ]
)

print(
    "Train accessed                :",
    training_metadata[
        "dataset_access"
    ][
        "train"
    ]
)

print(
    "Validation accessed           :",
    training_metadata[
        "dataset_access"
    ][
        "validation"
    ]
)

print(
    "Test accessed during training :",
    training_metadata[
        "dataset_access"
    ][
        "test"
    ]
)

print(
    "Initial checkpoint hash valid :",
    initial_hash_ok
)

print(
    "Best checkpoint hash valid    :",
    best_hash_ok
)

print(
    "\nTRAINING_METADATA_EXACT:",
    TRAINING_METADATA_EXACT
)


In [ ]:
# ============================================================
# 08.4 — SINGLE FROZEN TEST EVALUATION
# ============================================================

EXPECTED_TEST_COMPARISON = [
    {
        "model": "CONSTANT_TRAIN_MEAN",
        "source_cell": 139,
        "parameters": 0.0,
        "test_mse_standardized": 0.65235871,
        "kp_rmse": 0.42339761,
        "ki_rmse": 1.6803417,
        "regime_accuracy": 0.5,
        "rank_by_test_mse": 1
    },
    {
        "model": "CLASSICAL_LSTM",
        "source_cell": 139,
        "parameters": 5426.0,
        "test_mse_standardized": 0.86862868,
        "kp_rmse": 0.28910141,
        "ki_rmse": 4.15730145,
        "regime_accuracy": 0.5,
        "rank_by_test_mse": 2
    },
    {
        "model": "ENDPOINT_LINEAR",
        "source_cell": 139,
        "parameters": NaN,
        "test_mse_standardized": 1.035568,
        "kp_rmse": 0.21096838,
        "ki_rmse": 5.0409923,
        "regime_accuracy": 0.5,
        "rank_by_test_mse": 3
    },
    {
        "model": "ENDPOINT_MLP",
        "source_cell": 142,
        "parameters": 5554.0,
        "test_mse_standardized": 2.555413723,
        "kp_rmse": 0.38582,
        "ki_rmse": 7.70092,
        "regime_accuracy": 0.003125,
        "rank_by_test_mse": 4
    },
    {
        "model": "QLSTM",
        "source_cell": 149,
        "parameters": 5422.0,
        "test_mse_standardized": 2.963549411758347,
        "kp_rmse": 0.5153392859328884,
        "ki_rmse": 7.789277742495817,
        "regime_accuracy": 0.016875,
        "rank_by_test_mse": 5
    }
]


TEST_COMPARISON_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_QLSTM_RELPATHS[
        "test_comparison"
    ]
)


test_comparison = pd.read_csv(
    TEST_COMPARISON_PATH
)


expected_test_comparison = pd.DataFrame(
    EXPECTED_TEST_COMPARISON
)


TEST_COMPARISON_EXACT = bool(
    test_comparison.shape
    ==
    expected_test_comparison.shape

    and

    test_comparison[
        "model"
    ].tolist()
    ==
    expected_test_comparison[
        "model"
    ].tolist()

    and

    np.allclose(
        test_comparison[
            "test_mse_standardized"
        ].to_numpy(
            dtype=float
        ),
        expected_test_comparison[
            "test_mse_standardized"
        ].to_numpy(
            dtype=float
        ),
        rtol=0.0,
        atol=1e-12,
    )
)


if not TEST_COMPARISON_EXACT:

    raise RuntimeError(
        "Frozen single-test comparison changed."
    )


qlstm = (
    test_comparison.loc[
        test_comparison[
            "model"
        ]
        ==
        "QLSTM"
    ]
    .iloc[0]
)


lstm = (
    test_comparison.loc[
        test_comparison[
            "model"
        ]
        ==
        "CLASSICAL_LSTM"
    ]
    .iloc[0]
)


constant = (
    test_comparison.loc[
        test_comparison[
            "model"
        ]
        ==
        "CONSTANT_TRAIN_MEAN"
    ]
    .iloc[0]
)


qlstm_worse_vs_lstm_pct = (
    (
        float(
            qlstm[
                "test_mse_standardized"
            ]
        )
        -
        float(
            lstm[
                "test_mse_standardized"
            ]
        )
    )
    /
    float(
        lstm[
            "test_mse_standardized"
        ]
    )
    *
    100.0
)


qlstm_worse_vs_constant_pct = (
    (
        float(
            qlstm[
                "test_mse_standardized"
            ]
        )
        -
        float(
            constant[
                "test_mse_standardized"
            ]
        )
    )
    /
    float(
        constant[
            "test_mse_standardized"
        ]
    )
    *
    100.0
)


print(
    test_comparison.to_string(
        index=False
    )
)


print(
    "\nQLSTM test MSE                  :",
    float(
        qlstm[
            "test_mse_standardized"
        ]
    )
)

print(
    "QLSTM Kp RMSE                  :",
    float(
        qlstm[
            "kp_rmse"
        ]
    )
)

print(
    "QLSTM Ki RMSE                  :",
    float(
        qlstm[
            "ki_rmse"
        ]
    )
)

print(
    "QLSTM regime accuracy          :",
    float(
        qlstm[
            "regime_accuracy"
        ]
    )
)

print(
    "QLSTM rank                     :",
    int(
        qlstm[
            "rank_by_test_mse"
        ]
    )
)

print(
    "QLSTM MSE increase vs LSTM [%] :",
    qlstm_worse_vs_lstm_pct
)

print(
    "QLSTM increase vs constant [%] :",
    qlstm_worse_vs_constant_pct
)


QLSTM_SUPERIORITY_SUPPORTED = False

QUANTUM_ADVANTAGE_CLAIM_ALLOWED = False

POST_TEST_QLSTM_TUNING_ALLOWED = False


print(
    "\nQLSTM superiority supported    :",
    QLSTM_SUPERIORITY_SUPPORTED
)

print(
    "Quantum advantage claim allowed :",
    QUANTUM_ADVANTAGE_CLAIM_ALLOWED
)

print(
    "Post-test QLSTM tuning allowed  :",
    POST_TEST_QLSTM_TUNING_ALLOWED
)


In [ ]:
# ====================================================
# 08.5 — COMPUTATIONAL CONTEXT
# ====================================================

cell146_context = architecture[
    "cell146_context"
]


print(
    "Batch 256 feasible:",
    cell146_context[
        "batch_256_feasible"
    ]
)

print(
    "Estimated epoch [min]:",
    cell146_context[
        "estimated_epoch_min"
    ]
)

print(
    "QNode forward invocations / epoch:",
    cell146_context[
        "qnode_invocations_epoch_forward"
    ]
)

print(
    "Observed full training runtime [s]:",
    training_metadata[
        "training"
    ][
        "runtime_s"
    ]
)


print(
    "\nInterpretation:"
)

print(
    "These values document CPU simulation burden."
)

print(
    "They are NOT evidence of quantum speedup."
)

print(
    "Runtime comparison with float32 classical models "
    "is not treated as a controlled efficiency test."
)


QUANTUM_SPEEDUP_CLAIM_ALLOWED = False


In [ ]:
# ====================================================
# 08.6 — HISTORICAL IMPLEMENTATION SOURCE ARCHIVE
# ====================================================

snapshot = json.loads(
    RUNTIME_SOURCE_SNAPSHOT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_labels = [
    "146",
    "147",
    "148",
    "149",
    "150",
    "150-FIX",
]


SOURCE_ARCHIVE_COMPLETE = bool(
    snapshot[
        "labels"
    ]
    ==
    expected_labels

    and

    all(
        label
        in
        snapshot[
            "cells"
        ]

        for label
        in
        expected_labels
    )
)


print(
    "Archived runtime cells:"
)


for label in expected_labels:

    record = (
        snapshot[
            "cells"
        ][
            label
        ]
    )


    print(
        f"CELL {label:7s} | "
        f"{record['source_chars']:7d} chars | "
        f"{record['source_sha256']}"
    )


print(
    "\nSOURCE_ARCHIVE_COMPLETE:",
    SOURCE_ARCHIVE_COMPLETE
)


if not SOURCE_ARCHIVE_COMPLETE:

    raise RuntimeError(
        "QLSTM historical implementation archive incomplete."
    )


In [ ]:
# ====================================================
# 08.7 — EXPLICIT QLSTM RETRAINING GUARD
# ====================================================
#
# IMPORTANT:
#
# The exact Cell-147/148 implementation is archived,
# but src/models/qlstm.py is still a repository
# placeholder.
#
# We therefore DO NOT silently reconstruct a new
# implementation and call it scientifically identical.
#
# ====================================================

RETRAIN_QLSTM = False

QLSTM_RETRAINING_PERFORMED = False

MODEL_INFERENCE_PERFORMED = False


if not RETRAIN_QLSTM:

    print(
        "RETRAIN_QLSTM = False"
    )

    print(
        "Frozen Cell-148 checkpoint remains authoritative."
    )

    print(
        "No QLSTM training or inference was performed."
    )


else:

    raise RuntimeError(
        "Explicit QLSTM retraining is intentionally "
        "disabled in the migrated verification notebook. "
        "Use the archived Cell-146–150 source to first "
        "reconstruct and independently validate the exact "
        "QLSTM implementation in a controlled reproduction "
        "workflow. Do NOT rebuild or tune the model in "
        "response to the frozen test result."
    )


## 3. Handoff ke `09_closed_loop.ipynb`

Model-stage development berakhir setelah frozen test evaluation.

Handoff ke closed-loop menggunakan:

- frozen LSTM checkpoint;
- frozen QLSTM checkpoint;
- feature/target scaler yang sama;
- dataset/split yang sudah dibekukan;
- **tanpa model retraining setelah test**.

Kinerja regresi gain tidak dianggap ekuivalen dengan kinerja
pengendalian frekuensi. Karena itu hasil model harus diuji sebagai
gain scheduler pada plant closed-loop yang sama.

Evaluasi tersebut dilakukan di `09_closed_loop.ipynb`.


In [ ]:
# ====================================================
# 08.8 — QLSTM MODEL-STAGE SUMMARY
# ====================================================

NOTEBOOK_08_QLSTM_STAGE_READY = all(
    [
        FROZEN_QLSTM_ARTIFACTS_VALID,
        ARCHITECTURE_EXACT,
        TRAINING_METADATA_EXACT,
        initial_hash_ok,
        best_hash_ok,
        TEST_COMPARISON_EXACT,
        SOURCE_ARCHIVE_COMPLETE,
        not QLSTM_SUPERIORITY_SUPPORTED,
        not QUANTUM_ADVANTAGE_CLAIM_ALLOWED,
        not POST_TEST_QLSTM_TUNING_ALLOWED,
        not QUANTUM_SPEEDUP_CLAIM_ALLOWED,
        not QLSTM_RETRAINING_PERFORMED,
        not MODEL_INFERENCE_PERFORMED,
    ]
)


print("=" * 72)
print("08_train_qlstm.ipynb — SUMMARY")
print("=" * 72)


print(
    "Input shape                      : (11, 4)"
)

print(
    "Qubits                           :",
    arch[
        "n_qubits"
    ]
)

print(
    "VQC depth                        :",
    arch[
        "vqc_depth"
    ]
)

print(
    "QLSTM parameters                 :",
    parameter_audit[
        "qlstm_parameters"
    ]
)

print(
    "Quantum parameters               :",
    parameter_audit[
        "quantum_parameters"
    ]
)

print(
    "Best epoch                       :",
    training_metadata[
        "training"
    ][
        "best_epoch"
    ]
)

print(
    "Validation MSE                   :",
    training_metadata[
        "training"
    ][
        "best_validation_mse"
    ]
)

print(
    "Test MSE                         :",
    float(
        qlstm[
            "test_mse_standardized"
        ]
    )
)

print(
    "QLSTM superiority                : False"
)

print(
    "Quantum advantage claim          : False"
)

print(
    "Quantum speedup claim            : False"
)

print(
    "Post-test tuning                 : False"
)

print(
    "Test used for model selection    : False"
)

print(
    "Runtime source archive complete  :",
    SOURCE_ARCHIVE_COMPLETE
)

print(
    "QLSTM retraining requested       :",
    RETRAIN_QLSTM
)

print(
    "QLSTM retraining performed       :",
    QLSTM_RETRAINING_PERFORMED
)

print(
    "Model inference in verify mode   :",
    MODEL_INFERENCE_PERFORMED
)

print(
    "NOTEBOOK 08 QLSTM STAGE READY    :",
    NOTEBOOK_08_QLSTM_STAGE_READY
)


if NOTEBOOK_08_QLSTM_STAGE_READY:

    print(
        "\nNEXT NOTEBOOK:"
    )

    print(
        "09_closed_loop.ipynb"
    )
